In [2]:
import random
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from faker import Faker

In [3]:
"""
generate_retail_dataset.py
---------------------------------------------------------


Intentionally injected problems:
    - duplicate rows (same user_id + transaction_date)
    - missing values (NaN) in price, status, email, raw_timestamp
    - blank strings ("") used instead of NaN in text columns
    - invalid email formats
    - out-of-range ages
    - negative prices / sale amounts
    - inconsistent capitalization (region, status)
    - leading/trailing whitespace (city)
    - multiple date string formats (transaction_date)

Output: retail_dataset.csv  (1000 rows, 15 columns)
---------------------------------------------------------
"""

import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker

# ============================================================
# 1. SETUP
# ============================================================
random.seed(42)
np.random.seed(42)
fake = Faker()
Faker.seed(42)

TOTAL_ROWS = 1000
NUM_DUPLICATE_ROWS = 30                            # falls inside the 25-40 range asked for
NUM_BASE_ROWS = TOTAL_ROWS - NUM_DUPLICATE_ROWS    # rows we generate "fresh" (970)
NUM_USERS = 260                                    # pool of distinct customers

# ============================================================
# 2. REFERENCE / LOOKUP DATA
# ============================================================
# city -> region mapping, kept internally consistent on purpose
CITY_TO_REGION = {
    "Delhi": "North",
    "Chandigarh": "North",
    "Lucknow": "North",
    "Jaipur": "North",
    "Mumbai": "West",
    "Pune": "West",
    "Ahmedabad": "West",
    "Bangalore": "South",
    "Chennai": "South",
    "Hyderabad": "South",
    "Kolkata": "East",
    "Patna": "East",
    "Bhubaneswar": "East",
}
CITIES = list(CITY_TO_REGION.keys())

# Give Mumbai extra weight so at least one city ends up with MORE than
# 100 rows (needed for the "groupBy count > 100" exercise).
CITY_WEIGHTS = [0.18 if c == "Mumbai" else 0.82 / (len(CITIES) - 1) for c in CITIES]

SUBSCRIPTIONS = ["Premium", "Basic", "Free"]
STATUSES = ["Completed", "Pending", "Cancelled", "Returned"]
CATEGORIES = ["Electronics", "Grocery", "Clothing", "Furniture", "Beauty", "Sports", "Books"]
STORE_IDS = [f"STORE{i:03d}" for i in range(1, 6)]           # STORE001 .. STORE005
EMAIL_DOMAINS = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com", "rediffmail.com"]

USER_POOL = [f"USER{i:04d}" for i in range(1, NUM_USERS + 1)]

DATE_START = datetime(2025, 1, 1).date()
DATE_END = datetime(2026, 12, 31).date()


def random_date(start, end):
    """Pick a random date between two datetime.date objects (inclusive)."""
    span = (end - start).days
    return start + timedelta(days=random.randint(0, span))


# ============================================================
# 3. BUILD THE "CLEAN" BASE ROWS
# ============================================================
rows = []

for i in range(1, NUM_BASE_ROWS + 1):
    username = fake.user_name()

    t_date = random_date(DATE_START, DATE_END)
    t_time = timedelta(
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59),
    )
    raw_timestamp = datetime.combine(t_date, datetime.min.time()) + t_time

    city = random.choices(CITIES, weights=CITY_WEIGHTS, k=1)[0]

    rows.append(
        {
            "transaction_id": f"TXN{i:06d}",
            "user_id": random.choice(USER_POOL),
            "username": username,
            "email": f"{username}@{random.choice(EMAIL_DOMAINS)}",
            "age": random.randint(18, 70),
            "subscription": random.choice(SUBSCRIPTIONS),
            "city": city,
            "region": CITY_TO_REGION[city],
            "store_id": random.choice(STORE_IDS),
            "transaction_date": t_date,                # still a date object for now
            "raw_timestamp": raw_timestamp,             # still a datetime object for now
            "product_category": random.choice(CATEGORIES),
            "sale_amount": round(random.uniform(100, 100000), 2),
            "price": round(random.uniform(100, 100000), 2),
            "status": random.choice(STATUSES),
        }
    )

df = pd.DataFrame(rows)
n = len(df)   # 970


def sample_disjoint(exclude, k):
    """Randomly choose k row positions that are NOT already in `exclude`.
    Used so two different 'messy' rules never accidentally land on the
    same row and cancel each other out."""
    candidates = np.setdiff1d(np.arange(n), exclude)
    return np.random.choice(candidates, size=k, replace=False)


# ============================================================
# 4. INJECT REALISTIC "DIRTY DATA" PROBLEMS
# ============================================================

# ---- 4a. Missing values (~8% each) --------------------------------------
null_price_idx = np.random.choice(n, size=int(n * 0.08), replace=False)
null_status_idx = np.random.choice(n, size=int(n * 0.08), replace=False)
null_email_idx = np.random.choice(n, size=int(n * 0.08), replace=False)
null_ts_idx = np.random.choice(n, size=int(n * 0.08), replace=False)

df.loc[null_price_idx, "price"] = np.nan
df.loc[null_status_idx, "status"] = np.nan
df.loc[null_email_idx, "email"] = np.nan
df.loc[null_ts_idx, "raw_timestamp"] = pd.NaT

# ---- 4b. Negative values (~10 each), kept separate from the NaN rows ----
neg_price_idx = sample_disjoint(null_price_idx, 10)
neg_sale_idx = np.random.choice(n, size=10, replace=False)

df.loc[neg_price_idx, "price"] = -df.loc[neg_price_idx, "price"]
df.loc[neg_sale_idx, "sale_amount"] = -df.loc[neg_sale_idx, "sale_amount"]

# ---- 4c. Invalid email formats (~10), separate from the NaN emails ------
invalid_email_idx = sample_disjoint(null_email_idx, 10)
invalid_emails = [
    "not-an-email", "user@@gmail..com", "plainaddress", "missing_at_sign.com",
    "user@.com", "@nodomain.com", "user name@gmail.com", "user@domain,com",
    "user@domain..com", "user.gmail.com",
]
for idx, bad_email in zip(invalid_email_idx, invalid_emails):
    df.loc[idx, "email"] = bad_email

# ---- 4d. Blank username strings (~20) ------------------------------------
blank_username_idx = np.random.choice(n, size=20, replace=False)
df.loc[blank_username_idx, "username"] = ""

# ---- 4e. Age outliers: ~10 below 18, ~10 above 100 -----------------------
young_idx = np.random.choice(n, size=10, replace=False)
old_idx = sample_disjoint(young_idx, 10)
df.loc[young_idx, "age"] = np.random.randint(1, 18, size=10)
df.loc[old_idx, "age"] = np.random.randint(101, 116, size=10)

# ---- 4f. Mixed capitalization: region (West / west / WEST) --------------
case_region_idx = np.random.choice(n, size=int(n * 0.20), replace=False)
half = len(case_region_idx) // 2
df.loc[case_region_idx[:half], "region"] = df.loc[case_region_idx[:half], "region"].str.lower()
df.loc[case_region_idx[half:], "region"] = df.loc[case_region_idx[half:], "region"].str.upper()

# ---- 4g. Mixed capitalization: status (Pending / pending / PENDING) -----
non_null_status_idx = df.index[df["status"].notna()].to_numpy()
case_status_idx = np.random.choice(
    non_null_status_idx, size=int(len(non_null_status_idx) * 0.20), replace=False
)
half = len(case_status_idx) // 2
df.loc[case_status_idx[:half], "status"] = df.loc[case_status_idx[:half], "status"].str.lower()
df.loc[case_status_idx[half:], "status"] = df.loc[case_status_idx[half:], "status"].str.upper()

# ---- 4h. Leading/trailing spaces in city (~6%) ---------------------------
spacey_city_idx = np.random.choice(n, size=int(n * 0.06), replace=False)
spacing_options = ["  {}", "{}  ", "  {}  ", " {}"]
df.loc[spacey_city_idx, "city"] = [
    random.choice(spacing_options).format(c) for c in df.loc[spacey_city_idx, "city"]
]

# ---- 4i. Blank strings instead of NaN, in a few more text columns -------
blank_city_idx = sample_disjoint(spacey_city_idx, 5)
blank_sub_idx = np.random.choice(n, size=5, replace=False)
blank_cat_idx = np.random.choice(n, size=5, replace=False)

df.loc[blank_city_idx, "city"] = ""
df.loc[blank_sub_idx, "subscription"] = ""
df.loc[blank_cat_idx, "product_category"] = ""

# ============================================================
# 5. FORMAT DATE / TIMESTAMP COLUMNS (with inconsistent formats)
# ============================================================
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y"]


def format_date_mixed(d):
    """Most dates use ISO format, but some use other common formats
    so Spark's inferSchema will struggle with this column."""
    fmt = random.choices(DATE_FORMATS, weights=[0.7, 0.2, 0.1])[0]
    return d.strftime(fmt)


def format_timestamp(ts):
    if pd.isna(ts):
        return np.nan
    return ts.strftime("%Y-%m-%d %H:%M:%S")


df["transaction_date"] = df["transaction_date"].apply(format_date_mixed)
df["raw_timestamp"] = df["raw_timestamp"].apply(format_timestamp)

# ============================================================
# 6. CREATE DUPLICATE ROWS (same user_id + transaction_date, on purpose)
# ============================================================
dup_positions = np.random.choice(df.index, size=NUM_DUPLICATE_ROWS, replace=False)
duplicate_rows = df.loc[dup_positions].copy()

df_final = pd.concat([df, duplicate_rows], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle rows

# Put the columns back in the exact order required
COLUMN_ORDER = [
    "transaction_id", "user_id", "username", "email", "age", "subscription",
    "city", "region", "store_id", "transaction_date", "raw_timestamp",
    "product_category", "sale_amount", "price", "status",
]
df_final = df_final[COLUMN_ORDER]

# ============================================================
# 7. SAVE + QUICK LOOK
# ============================================================
df_final.to_csv("retail_dataset.csv", index=False)

print("First 5 rows:")
print(df_final.head())
print("\nDataset shape:", df_final.shape)

# ---- Optional sanity check, handy before you open Spark -----------------
print("\n--- Quick data-quality summary (for reference) ---")
print("Missing values per column:\n", df_final.isna().sum())
print("\nExact duplicate rows:", df_final.duplicated().sum())
print("\nRows per city (top 5, ignoring spacing):\n",
      df_final["city"].str.strip().value_counts().head())

First 5 rows:
  transaction_id   user_id  username                 email  age subscription  \
0      TXN000522  USER0091  shawgary         @nodomain.com   68      Premium   
1      TXN000738  USER0204  sheryl79  sheryl79@outlook.com   18      Premium   
2      TXN000741  USER0064    vbrock                   NaN   22        Basic   
3      TXN000661  USER0209  trussell                   NaN   48      Premium   
4      TXN000412  USER0069   nwarren   nwarren@outlook.com   20        Basic   

         city region  store_id transaction_date        raw_timestamp  \
0       Patna   East  STORE004       2025-02-14  2025-02-14 03:47:03   
1     Patna     EAST  STORE004       2025-05-27                  NaN   
2  Chandigarh  North  STORE001       2026-10-14  2026-10-14 05:05:18   
3        Pune   WEST  STORE004       04-28-2025  2025-04-28 08:03:20   
4      Mumbai   West  STORE004       17/07/2025  2025-07-17 00:44:24   

  product_category  sale_amount     price     status  
0           Beaut

# Synthetic Retail Dataset Generation using LLM

# This notebook generates a synthetic retail transaction dataset for Apache Spark (PySpark) practice.

# The dataset generation code was created with the assistance of a **Large Language Model (LLM)** to simulate realistic retail transaction data. The generated dataset intentionally includes missing values, duplicate records, invalid entries, inconsistent formatting, and other real-world data quality issues. These characteristics make it suitable for practicing data cleaning, transformation, aggregation, and analysis using Apache Spark.

# The final dataset is saved as *`retail_dataset.csv`* and will be used in the subsequent Spark data processing tasks.

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [6]:
spark = SparkSession.builder.appName("CEI_Week_5_Assignment").getOrCreate()

In [7]:
df = spark.read.csv("retail_dataset.csv", header=True, inferSchema=True)

In [8]:
df = df.show(5)

+--------------+--------+--------+--------------------+---+------------+----------+------+--------+----------------+-------------------+----------------+-----------+--------+---------+
|transaction_id| user_id|username|               email|age|subscription|      city|region|store_id|transaction_date|      raw_timestamp|product_category|sale_amount|   price|   status|
+--------------+--------+--------+--------------------+---+------------+----------+------+--------+----------------+-------------------+----------------+-----------+--------+---------+
|     TXN000522|USER0091|shawgary|       @nodomain.com| 68|     Premium|     Patna|  East|STORE004|      2025-02-14|2025-02-14 03:47:03|          Beauty|    11694.9|    NULL| Returned|
|     TXN000738|USER0204|sheryl79|sheryl79@outlook.com| 18|     Premium|   Patna  |  EAST|STORE004|      2025-05-27|               NULL|     Electronics|    9777.38|77396.99| Returned|
|     TXN000741|USER0064|  vbrock|                NULL| 22|       Basic|Cha

# Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

### Answer

Traditional **MapReduce (Hadoop)** has several limitations that make **Apache Spark** the preferred framework for modern big data processing.

#### Limitations of MapReduce

1. **Disk-Based Processing**
    MapReduce writes intermediate results to disk after every Map and Reduce stage.
   
    This causes high disk I/O and increases processing time.

2. **Slower Performance**
   
    Because of frequent disk operations, MapReduce is slower, especially for large and iterative workloads.

3. **Complex Programming Model**
   
    Developers need to write separate Map and Reduce functions, making application development more difficult.

4. **Not Suitable for Iterative Processing**
   
    Machine learning and data analytics require repeated processing of the same data, but MapReduce reads data from disk during every iteration.

#### Why Apache Spark is Preferred

Apache Spark overcomes these limitations by:

- **In-Memory Computing:** Spark stores intermediate data in RAM instead of writing it to disk, significantly improving performance.
- **10× to 100× Faster than Hadoop MapReduce:** In-memory execution greatly reduces processing time for large datasets.
- **Simple APIs:** Spark provides easy-to-use APIs in Python, Java, Scala, and R, making development simpler and faster.
- **Support for Multiple Workloads:** Spark supports SQL, Machine Learning (MLlib), Streaming, and Graph Processing within a single framework.



Due to its *in-memory processing, higher speed (10×–100× faster than Hadoop MapReduce), simple APIs, and support for modern data processing workloads**, Apache Spark is the preferred choice for big data applications.

#Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.


### Answer

Apache Spark uses **In-Memory Computing**, which stores intermediate data in RAM instead of writing it to disk after every operation. This reduces disk I/O and significantly improves processing speed.

#### How Spark Speeds Up Iterative Machine Learning Algorithms

1. **In-Memory Processing**
   - Spark caches frequently used data in memory (RAM), allowing repeated access without reading from disk.

2. **Faster Iterations**
   - Machine learning algorithms process the same dataset multiple times. Spark reuses the cached data, making each iteration much faster.

3. **Reduced Disk I/O and Execution time**
   - Unlike Hadoop MapReduce, Spark avoids repeatedly reading and writing intermediate data to disk, reducing execution time.

4. **Higher Performance**
   - Spark can be **10× faster on disk-based workloads and up to 100× faster with in-memory processing** compared to Hadoop MapReduce.



By storing data in memory and minimizing disk access, Apache Spark provides faster and more efficient execution of iterative machine learning algorithms than traditional disk-based systems.

# Q3 : Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [8]:

df_rm_duplicates = df.dropDuplicates(["user_id", "transaction_date"])


In [10]:
#before removing duplicates count

df.count()

1000

In [11]:
#After removing duplicates count

df_rm_duplicates.count()

969

In [22]:
from pyspark.sql.functions import col

duplicates = (
    df.groupBy("user_id", "transaction_date")
      .count()
      .filter(col("count") > 1)
)

duplicates.count()

31

**Removed duplicate records from the DataFrame based on the combination of **user_id** and **transaction_date**.**


**Also  Verified that the duplicate rows have been removed successfully.**

# Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [33]:
avg_sale_amount = df.filter(df.region == "West").groupBy("product_category").avg("sale_amount")

In [34]:
avg_sale_amount.show()

+----------------+------------------+
|product_category|  avg(sale_amount)|
+----------------+------------------+
|          Sports|51858.245789473694|
|            NULL|          86471.82|
|         Grocery|44896.250499999995|
|     Electronics| 45953.91868421053|
|        Clothing|49761.052916666646|
|           Books| 48683.49058823529|
|       Furniture|52776.185384615375|
|          Beauty| 58672.42880000001|
+----------------+------------------+



# Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.



Difference between .na.drop() and .na.fill() is that

.na.drop() — Removes rows that contain null (missing) values. By default, it drops a row if any column has a null, but you can configure it to drop only when all columns are null, or only when nulls appear in specific columns.



.na.fill() — Replaces null values with a specified default value, keeping all rows intact. You can fill all columns with one value, or pass a dictionary to fill different columns with different values (e.g., fill missing price with 0 and missing status with "Unknown").

In [40]:
df_filling_nulls = df.na.fill({"status": "Unknown"})
df_filling_nulls.select("status").show(50)

+---------+
|   status|
+---------+
| Returned|
| Returned|
|Completed|
|Cancelled|
|  Unknown|
|Completed|
| Returned|
| Returned|
|Cancelled|
|Cancelled|
| Returned|
| Returned|
| Returned|
|Cancelled|
|Cancelled|
|Cancelled|
|Completed|
|Completed|
|  Unknown|
|Completed|
|Cancelled|
|Completed|
|Completed|
|Completed|
|  Unknown|
| Returned|
|Completed|
| Returned|
|  Unknown|
| Returned|
|  Pending|
|completed|
| RETURNED|
|Completed|
| Returned|
|Cancelled|
|Cancelled|
|  Pending|
| Returned|
|Cancelled|
|Cancelled|
|Cancelled|
|COMPLETED|
| Returned|
|  Pending|
| Returned|
|  PENDING|
|Completed|
|  Pending|
|  Pending|
+---------+
only showing top 50 rows


# Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [41]:
count_of_city = df.groupBy("city").count().filter("count > 100")



In [43]:
count_of_city.show()

+------+-----+
|  city|count|
+------+-----+
|Mumbai|  191|
+------+-----+



conclusion:

Grouped the records by **city**, counted the total number of records for each city, and displayed only the cities with **more than 100 records**.

# Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?bold text

Spark DataFrames are immutable which means once created, they can never be changed. Any operation you perform on a DataFrame (like dropping a column, renaming a column, filtering rows, or filling nulls) doesn't alter the original DataFrame at all.

 Instead, Spark takes the original data, applies your transformation, and hands you back a brand new DataFrame with the result. The original one stays exactly as it was.


for example we can  Think of a DataFrame like a printed photograph. If you want to crop it or add a filter, you can't edit the original print directly — you make a new copy with the changes, while the original photo stays untouched in the drawer.


i.e:
df2 = df.drop("email")     


 df2 is a NEW DataFrame without the 'email' column and df itself is completely unchanged



print(df.columns)    # still shows 'email'


print(df2.columns)    # does NOT show 'email'

# Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [45]:
premium_users = df.filter(df.age.between(18, 30) & (df.subscription == "Premium"))

In [46]:
premium_users.show()

+--------------+--------+---------------+--------------------+---+------------+-------------+------+--------+----------------+-------------------+----------------+-----------+--------+---------+
|transaction_id| user_id|       username|               email|age|subscription|         city|region|store_id|transaction_date|      raw_timestamp|product_category|sale_amount|   price|   status|
+--------------+--------+---------------+--------------------+---+------------+-------------+------+--------+----------------+-------------------+----------------+-----------+--------+---------+
|     TXN000738|USER0204|       sheryl79|sheryl79@outlook.com| 18|     Premium|      Patna  |  EAST|STORE004|      2025-05-27|               NULL|     Electronics|    9777.38|77396.99| Returned|
|     TXN000140|USER0252|         jrivas|    jrivas@yahoo.com| 25|     Premium|        Patna|  east|STORE001|      2025-08-08|2025-08-08 19:18:44|       Furniture|   17348.37|    NULL|completed|
|     TXN000880|USER0259|

# Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

Handling Null Values Before Aggregation:


It's better to handle null values before performing mathematical aggregations, because nulls can lead to inaccurate or incomplete results. By removing or replacing null values ahead of time, functions like sum() and avg() produce calculations you can actually trust — rather than numbers that look fine on the surface but are quietly based on incomplete data.


A simple analogy: Imagine calculating the average price of 10 products, but 3 of them have no price entered (null).

 If those nulls aren't handled first, Spark quietly leaves them out — so your "average of 10 products" is secretly just an average of 7.

 Cleaning the nulls first means you know exactly what you're calculating.



# Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [49]:
df = df.withColumn("event_time", to_timestamp("raw_timestamp"))


In [50]:
df = df.drop("raw_timestamp")

In [59]:
df.select("user_id","username","subscription","event_time").show()

+--------+--------------+------------+-------------------+
| user_id|      username|subscription|         event_time|
+--------+--------------+------------+-------------------+
|USER0091|      shawgary|     Premium|2025-02-14 03:47:03|
|USER0204|      sheryl79|     Premium|               NULL|
|USER0064|        vbrock|       Basic|2026-10-14 05:05:18|
|USER0209|      trussell|     Premium|2025-04-28 08:03:20|
|USER0069|       nwarren|       Basic|2025-07-17 00:44:24|
|USER0196|         ugill|        Free|2026-11-25 01:35:03|
|USER0213| jenniferburns|        Free|2025-01-09 03:40:54|
|USER0255|      coleerin|        Free|2026-08-29 02:52:52|
|USER0212| hatfieldsarah|        Free|2025-08-01 12:06:31|
|USER0082|          NULL|        Free|2025-03-18 21:40:25|
|USER0064|     candice25|     Premium|2025-06-08 07:03:34|
|USER0204|  richarddavid|        Free|2025-03-18 15:50:16|
|USER0126|       ybrooks|       Basic|2025-07-26 14:26:49|
|USER0104|  sandraromero|        Free|               NUL

# Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

A shuffle happens when Spark has to move data around between partitions so that related data ends up in the same place. This usually happens with operations like groupBy(), join(), because Spark can't group things together if they're scattered across different partitions — it first has to collect them in one spot.


It's called a wide transformation because the data isn't staying in its own partition, it's actually moving across partitions, and sometimes even across different machines in the cluster.

This movement takes extra time and uses more resources, which is why shuffles tend to slow things down compared to operations that don't need to move data at all.


**For example**: if I do a groupBy() on region, Spark needs to bring all the "North" rows together, all the "South" rows together, and so on — even if they were originally spread out across different partitions. That process of gathering everything together is basically what a shuffle is.

# Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [62]:
df.filter(df.email.isNull() | (df.username == "")).show()

+--------------+--------+----------------+-----+---+------------+-----------+------+--------+----------------+----------------+-----------+--------+---------+-------------------+
|transaction_id| user_id|        username|email|age|subscription|       city|region|store_id|transaction_date|product_category|sale_amount|   price|   status|         event_time|
+--------------+--------+----------------+-----+---+------------+-----------+------+--------+----------------+----------------+-----------+--------+---------+-------------------+
|     TXN000741|USER0064|          vbrock| NULL| 22|       Basic| Chandigarh| North|STORE001|      2026-10-14|          Beauty|   23554.53|48570.92|Completed|2026-10-14 05:05:18|
|     TXN000661|USER0209|        trussell| NULL| 48|     Premium|       Pune|  WEST|STORE004|      04-28-2025|          Sports|   23450.67|88626.97|Cancelled|2025-04-28 08:03:20|
|     TXN000939|USER0136|   deborahporter| NULL| 46|       Basic|     Mumbai|  West|STORE001|      2025-0

In [63]:
df = df.filter(df.email.isNotNull() & (df.username != ""))



In [64]:
df.show()

+--------------+--------+--------------+--------------------+---+------------+----------+------+--------+----------------+----------------+-----------+--------+---------+-------------------+
|transaction_id| user_id|      username|               email|age|subscription|      city|region|store_id|transaction_date|product_category|sale_amount|   price|   status|         event_time|
+--------------+--------+--------------+--------------------+---+------------+----------+------+--------+----------------+----------------+-----------+--------+---------+-------------------+
|     TXN000522|USER0091|      shawgary|       @nodomain.com| 68|     Premium|     Patna|  East|STORE004|      2025-02-14|          Beauty|    11694.9|    NULL| Returned|2025-02-14 03:47:03|
|     TXN000738|USER0204|      sheryl79|sheryl79@outlook.com| 18|     Premium|   Patna  |  EAST|STORE004|      2025-05-27|     Electronics|    9777.38|77396.99| Returned|               NULL|
|     TXN000412|USER0069|       nwarren| nwar

NOW IT CONTAIN ONLY THOSE ENTRIES WHERE E-mail and username exists.

# Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [67]:
df_mul_statistics = df.groupBy("product_category").agg( min("price").alias("minimum_price"), max("price").alias("maximum_price"), mean("price")).alias("mean_price")

In [68]:
df_mul_statistics.show()

+----------------+-------------+-------------+------------------+
|product_category|minimum_price|maximum_price|        avg(price)|
+----------------+-------------+-------------+------------------+
|          Sports|    -14830.02|     98440.52| 51934.71310344828|
|            NULL|      5185.57|     88150.63|55335.090000000004|
|         Grocery|    -32353.62|      99099.5| 52727.70317829459|
|     Electronics|    -34423.06|      99845.6|    49246.93796875|
|        Clothing|    -14642.42|     99872.94| 51685.45274193544|
|           Books|    -64752.08|     99342.32|52474.980970873796|
|       Furniture|    -46976.32|     98928.54| 49781.90794642854|
|          Beauty|    -22470.55|     99663.48| 50765.89913043479|
+----------------+-------------+-------------+------------------+



Shows multiple statistics at once

# Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

Risk of Using inferSchema=True:


When I set inferSchema=True, Spark looks at the actual data in each column and tries to guess the right data type on its own, instead of me telling it upfront.

The problem is, if a column like transaction_date has messy or inconsistent formats — like some dates written as 2025-06-12 and others as 12/06/2025 — Spark gets confused and usually just falls back to treating the whole column as a string, instead of a proper date or timestamp.

That might not sound like a big deal, but it causes problems later. If the column is read as a string, I can't directly use date functions on it, sorting won't work the way I expect, and if I try to cast it to a date type afterward, some of the messy-format rows might just turn into nulls instead of throwing an error — so I might not even notice the data is broken.


Basically, inferSchema is convenient, but it's guessing based on what it sees, and if the data isn't clean and consistent, its guess can be wrong — which is why it's often safer to define the schema manually for important columns.

# Q15: Write a final processing pipeline that:

# Filters out duplicates.

# Fills null prices with 0.

# Groups by store_id to calculate total revenue.

In [83]:
pipeline = df.dropDuplicates()

pipeline = pipeline.na.fill({"price": 0})

pipeline = pipeline.groupBy("store_id").sum("price").withColumnRenamed("sum(price)", "total_revenue")

In [84]:
pipeline.show()

+--------+-----------------+
|store_id|    total_revenue|
+--------+-----------------+
|STORE004|8005179.470000002|
|STORE003|8136923.549999998|
|STORE001|8351974.509999998|
|STORE002|7513939.789999999|
|STORE005|       9389246.82|
+--------+-----------------+



# Conclusion

The dataset was successfully processed using Apache Spark.

 Duplicate records were removed, missing values in the **price** column are replaced with **0**, and the data was grouped by **store_id** to calculate the total revenue for each store. This pipeline demonstrates how Spark can efficiently clean and transform data for analysis.